# CosmX Pancreas Evaluation

In [3]:
import scanpy as sc
import numpy as np
import pandas as pd

import InterScale as interscale
from InterScale.config import load_config
from InterScale.tl import prepare_geome_dataset, check_and_update_cfg
from InterScale.geome_dataloader import GraphAnnDataModule
#from InterScale.eval.gene_rank_analysis import predict_gene_r2, gene_rank_analysis
from InterScale.tl import prepare_a2d_dataset

from pathlib import Path
import torch

In [4]:
CFG_CLASS = "/dss/dsshome1/05/di93tig/1_projects/GT-long-range-niches/src/config_files/Cosmx_pancreas/clas_graph.yaml"

## Load pretrained models

In [ ]:
RESULTS_DIR = '/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/results/cosmx_pancreas/model/'

In [5]:
CFG = "/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/src/config_files/CosmX_Pancreas/pancreas_regr_sw_CombinedComponent.yaml"
cfg = load_config(CFG_CLASS)

In [9]:
cfg.dataset.layer_key == ""

True

In [ ]:
adata = sc.read_h5ad(cfg.dataset.h5ad_data)
adata

In [ ]:
adata.obsm['spatial']

In [ ]:
# import torch

# model = torch.load('/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/results/cosmx_pancreas/pancreas_regr_node_GCN__model.pt')
# state_dict = model['state_dict']

In [ ]:
interscale.model.LocalModel._setup_anndata(adata = adata, prediction_task = cfg.dataset.prediction_task, layer_key = cfg.dataset.layer_key, sample_key_list = cfg.dataset.sample_key, prediction_obs =  cfg.dataset.prediction_obs, group_key = cfg.dataset.group_label, view_registry = False)
local_model = interscale.model.LocalModel.load(RESULTS_DIR, adata, cfg, local_component = True, global_component = False, wandb_save = True)
interscale.model.CombinedModel._setup_anndata(adata = adata, prediction_task = cfg.dataset.prediction_task, layer_key = cfg.dataset.layer_key, sample_key_list = cfg.dataset.sample_key, prediction_obs =  cfg.dataset.prediction_obs, group_key = cfg.dataset.group_label, view_registry = False)
combined_model = interscale.model.CombinedModel.load(RESULTS_DIR, adata, cfg, local_component = True, global_component = True, wandb_save = True)

## Inference 

In [ ]:
sub_adata = adata[adata.obs['condition'] == 'T1D']

In [ ]:
result = local_model.get_model_output(sub_adata, prefix = 'local')

In [ ]:
result = combined_model.get_model_output(result, prefix = 'combined')
result

## Gene Rank analysis

In [ ]:
gene_rank_analysis(result,
                   layers_local_pred = 'local_y_pred',
                   layers_global_pred = 'combined_y_pred',
                   top_n = 5,
                   plot_result = True,
                   return_top_genes = True)